In [0]:
SRC  = "/Volumes/project_lakehouse_telemetria_ele/bronze/landing/telemetria/"
CHK  = "/Volumes/project_lakehouse_telemetria_ele/bronze/landing/_checkpoints/telemetria"
TBL  = "project_lakehouse_telemetria_ele.bronze.telemetria_raw"

from pyspark.sql import functions as F

(spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.schemaLocation", CHK + "/schema")
  .option("cloudFiles.inferColumnTypes", "false")      # Bronze = tudo bruto, tipagem fica pra Silver
  .load(SRC)
  .withColumn("_ingest_ts", F.current_timestamp())
  .withColumn("_source_file", F.col("_metadata.file_path"))
  .writeStream
  .option("checkpointLocation", CHK + "/ckpt")
  .trigger(availableNow=True)
  .toTable(TBL))